# Module 1: Building a Multi-Agent Customer Service System

## Overview

In this module, you'll build a production-ready multi-agent system for e-commerce customer service using the Strands Agent SDK with **MCP (Model Context Protocol)** tools. 

### MCP Architecture
The Order and Product agents use **MCP (Model Context Protocol)** to connect to separate MCP servers that expose tools. This provides:
- **Decoupling**: Tools run as separate processes
- **Reusability**: MCP servers can be shared across applications
- **Standardization**: Follows the MCP protocol for tool discovery

### Learning Objectives
1. Build specialized agents with Strands SDK and MCP tools
2. Create an orchestrator for intelligent request routing
3. Understand cost optimization with mixed LLM strategy
4. Test the multi-agent system with various queries


## Step 1: Environment Setup

First, let's set up our environment and verify we have access to the required resources.

In [ ]:
# Install dependencies if needed
!pip install -q strands-agents strands-agents-tools boto3 mcp

In [ ]:
import boto3
import json
import os
import sys
from datetime import datetime

# Get AWS region
session = boto3.Session()
REGION = session.region_name or 'us-west-2'
print(f"AWS Region: {REGION}")

# Set environment variables for tools
os.environ['AWS_REGION'] = REGION

# Get table names from SSM (set by workshop infrastructure)
ssm = boto3.client('ssm', region_name=REGION)

try:
    os.environ['ORDERS_TABLE_NAME'] = ssm.get_parameter(Name='ecommerce-workshop-orders-table')['Parameter']['Value']
    os.environ['ACCOUNTS_TABLE_NAME'] = ssm.get_parameter(Name='ecommerce-workshop-accounts-table')['Parameter']['Value']
    os.environ['PRODUCTS_TABLE_NAME'] = ssm.get_parameter(Name='ecommerce-workshop-products-table')['Parameter']['Value']
    print(f"✓ Orders Table: {os.environ['ORDERS_TABLE_NAME']}")
    print(f"✓ Accounts Table: {os.environ['ACCOUNTS_TABLE_NAME']}")
    print(f"✓ Products Table: {os.environ['PRODUCTS_TABLE_NAME']}")
except Exception as e:
    print(f"Note: Could not retrieve SSM parameters ({e})")
    print("Using default table names - ensure infrastructure is deployed")
    os.environ['ORDERS_TABLE_NAME'] = 'ecommerce-workshop-orders'
    os.environ['ACCOUNTS_TABLE_NAME'] = 'ecommerce-workshop-accounts'
    os.environ['PRODUCTS_TABLE_NAME'] = 'ecommerce-workshop-products'

## Step 2: Understanding MCP Tools

The Order and Product agents use **MCP (Model Context Protocol)** to access tools. Instead of importing Python functions directly, the agents connect to MCP servers that expose tools.

### MCP Server Architecture

```
Agent (Haiku)  ──MCPClient──►  MCP Server  ──►  DynamoDB
                  (stdio)      (FastMCP)
```

### MCP Servers in this Module

| Server | File | Tools |
|--------|------|-------|
| Order Service | `mcp_servers/order_mcp_server.py` | get_order_status, track_shipment, process_return, modify_order, get_customer_orders |
| Product Service | `mcp_servers/product_mcp_server.py` | search_products, get_product_details, check_inventory, get_product_recommendations, compare_products, get_return_policy |

### How Agents Connect to MCP

```python
from strands.tools.mcp import MCPClient
from mcp import StdioServerParameters
from mcp.client.stdio import stdio_client

# Configure MCP server connection
server_params = StdioServerParameters(
    command="python",
    args=["mcp_servers/order_mcp_server.py"]
)

# Connect and get tools
mcp_client = MCPClient(lambda: stdio_client(server_params))
mcp_client.__enter__()
tools = mcp_client.list_tools_sync()

# Create agent with MCP tools
agent = Agent(model=model, tools=tools)
```

In [ ]:
# Verify MCP servers exist
# Add agents to path
sys.path.insert(0, 'agents')
import os
from pathlib import Path

mcp_servers_dir = Path('mcp_servers')
order_server = mcp_servers_dir / 'order_mcp_server.py'
product_server = mcp_servers_dir / 'product_mcp_server.py'

print("MCP Servers:")
print(f"  Order Server: {order_server} - {'✓ exists' if order_server.exists() else '✗ missing'}")
print(f"  Product Server: {product_server} - {'✓ exists' if product_server.exists() else '✗ missing'}")

## Step 3: Build Specialized Agents

Now let's create our specialized agents. The Order and Product agents use **MCP tools** from separate MCP servers, while the Account agent uses native tools.

The MCP servers are located in `mcp_servers/` directory:
- `order_mcp_server.py` - Exposes order management tools
- `product_mcp_server.py` - Exposes product catalog tools

In [ ]:
from strands import Agent
from strands.models import BedrockModel

# Model configuration using global cross-region inference profiles
HAIKU_MODEL_ID = "global.anthropic.claude-haiku-4-5-20251001-v1:0"
SONNET_MODEL_ID = "global.anthropic.claude-sonnet-4-5-20250929-v1:0"

# Pricing per 1M tokens (approximate, check AWS pricing page for latest)
# https://aws.amazon.com/bedrock/pricing/
HAIKU_INPUT_PRICE = 0.80    # $0.80 per 1M input tokens
HAIKU_OUTPUT_PRICE = 4.00   # $4.00 per 1M output tokens
SONNET_INPUT_PRICE = 3.00   # $3.00 per 1M input tokens  
SONNET_OUTPUT_PRICE = 15.00 # $15.00 per 1M output tokens

print(f"Haiku Model: {HAIKU_MODEL_ID}")
print(f"Sonnet Model: {SONNET_MODEL_ID}")
print("\nOrder and Product agents use MCP tools via MCPClient")

### 3.1 Create Order Agent

In [ ]:
from order_agent import create_order_agent

order_agent = create_order_agent(region=REGION)
print(f"Order Agent created with model: {HAIKU_MODEL_ID}")
print("Tools loaded via MCP from order_mcp_server.py")

In [ ]:
# Test Order Agent directly
response = order_agent("What's the status of order ORD-2024-10002?")
print("Order Agent Response:")
print(response)

### 3.2 Create Product Agent

In [ ]:
from product_agent import create_product_agent

product_agent = create_product_agent(region=REGION)
print(f"Product Agent created with model: {HAIKU_MODEL_ID}")
print("Tools loaded via MCP from product_mcp_server.py")

In [ ]:
# Test Product Agent directly
response = product_agent("Do you have any wireless headphones under $100?")
print("Product Agent Response:")
print(response)

### 3.3 Create Account Agent

In [ ]:
from account_agent import create_account_agent

account_agent = create_account_agent(region=REGION)
print(f"Account Agent created with model: {HAIKU_MODEL_ID}")

In [ ]:
# Test Account Agent directly
response = account_agent("What are the benefits of Gold membership?")
print("Account Agent Response:")
print(response)

## Step 4: Create the Orchestrator

The Orchestrator uses Claude Sonnet 4.5 (larger model) for complex reasoning and routes requests to specialized agents.

In [ ]:
from orchestrator import MultiAgentCustomerService

# Create the multi-agent system
customer_service = MultiAgentCustomerService(region=REGION)
print("Multi-Agent Customer Service System initialized!")
print(f"- Orchestrator: {SONNET_MODEL_ID}")
print(f"- Order Agent: {HAIKU_MODEL_ID} (MCP tools)")
print(f"- Product Agent: {HAIKU_MODEL_ID} (MCP tools)")
print(f"- Account Agent: {HAIKU_MODEL_ID} (native tools)")

## Step 5: Test the Multi-Agent System

Let's test various customer scenarios to see how the orchestrator routes requests.

In [ ]:
# Test Case 1: Order inquiry (should route to Order Agent)
print("="*60)
print("Test Case 1: Order Inquiry")
print("="*60)

response = customer_service.chat("Where is my order ORD-2024-10002?")
print(f"Customer: Where is my order ORD-2024-10002?")
print(f"\nAgent: {response}")

In [ ]:
# Test Case 2: Product search (should route to Product Agent)
print("="*60)
print("Test Case 2: Product Search")
print("="*60)

response = customer_service.chat("I'm looking for a good gaming keyboard with RGB lighting")
print(f"Customer: I'm looking for a good gaming keyboard with RGB lighting")
print(f"\nAgent: {response}")

In [ ]:
# Test Case 3: Account management (should route to Account Agent)
print("="*60)
print("Test Case 3: Account Management")
print("="*60)

response = customer_service.chat("I need to reset my password for john.smith@email.com")
print(f"Customer: I need to reset my password for john.smith@email.com")
print(f"\nAgent: {response}")

In [ ]:
# Test Case 4: Complex multi-domain query (should use multiple agents)
print("="*60)
print("Test Case 4: Complex Multi-Domain Query")
print("="*60)

response = customer_service.chat(
    "I want to return order ORD-2024-10001 because the headphones don't fit well. "
    "Can you recommend a different pair that might be more comfortable?"
)
print(f"Customer: I want to return order ORD-2024-10001 because the headphones don't fit well. Can you recommend a different pair that might be more comfortable?")
print(f"\nAgent: {response}")

In [ ]:
# Test Case 5: Inventory check
print("="*60)
print("Test Case 5: Inventory Check")
print("="*60)

response = customer_service.chat("Is the 4K webcam PROD-088 in stock?")
print(f"Customer: Is the 4K webcam PROD-088 in stock?")
print(f"\nAgent: {response}")

## Summary

In this module, you built a multi-agent customer service system using **MCP tools** that:

1. **Uses MCP tools** - Order and Product agents connect to MCP servers for tool access
2. **Optimizes costs** by using smaller models for specialized tasks
3. **Routes intelligently** using a larger model for orchestration only
4. **Handles complex queries** that span multiple domains

### MCP Architecture

```
┌─────────────────────────────────────────────────────────────┐
│                    Orchestrator (Sonnet)                     │
│                   - Intent classification                    │
│                   - Request routing                          │
└─────────────────────┬───────────────┬───────────────────────┘
                      │               │
        ┌─────────────▼───┐   ┌───────▼─────────────┐
        │  Order Agent    │   │   Product Agent     │
        │   (Haiku)       │   │     (Haiku)         │
        │   MCP Tools     │   │     MCP Tools       │
        └────────┬────────┘   └─────────┬───────────┘
                 │                      │
        ┌────────▼────────┐   ┌─────────▼───────────┐
        │ order_mcp_server│   │ product_mcp_server  │
        │    (FastMCP)    │   │     (FastMCP)       │
        └────────┬────────┘   └─────────┬───────────┘
                 │                      │
        ┌────────▼────────┐   ┌─────────▼───────────┐
        │    DynamoDB     │   │     DynamoDB        │
        │  Orders Table   │   │   Products Table    │
        └─────────────────┘   └─────────────────────┘
```

### Key Files

| File | Description |
|------|-------------|
| `mcp_servers/order_mcp_server.py` | MCP server for order tools |
| `mcp_servers/product_mcp_server.py` | MCP server for product tools |
| `agents/order_agent.py` | Order Agent with MCP client |
| `agents/product_agent.py` | Product Agent with MCP client |
| `agents/orchestrator.py` | Multi-agent orchestrator |


### Next Steps

In **Module 2**, we'll evaluate this system with a comprehensive test dataset and establish baseline metrics for production monitoring.

In [ ]:
# Clean up MCP connections before ending
order_agent.cleanup()
product_agent.cleanup()
customer_service.cleanup()
print("MCP connections cleaned up")

# Save region for use in next module
%store REGION
print("Session data saved for Module 2!")